# Reconnaître un schéma d'argumentation — la table de Walton et un classifieur déterministe (C#)

**Notebook pédagogique — série ArgumentAnalysis, jumeau C#.** Ce carnet porte l'essence d'un organe du dépôt EPITA (mandat Triple Distillation) : la table des **schémas d'argumentation** de Walton et son **classifieur lexical déterministe**. La série dispose déjà du modèle *structurel* de Toulmin (`Toulmin_Model` : claim/data/warrant) et du cadre *abstrait* de Dung (`Dung_AF_Semantics` : arguments qui s'attaquent). Ici, le niveau intermédiaire : les **schémas stéréotypés** — autorité, analogie, cause à effet, consensus... — chacun avec ses prémisses types, sa conclusion type et surtout ses **questions critiques**, le test qu'un opposant applique pour prendre l'argument en défaut.

**Portage C#.** Le jumeau Python [`Argumentation-01b-Schemes-Walton-Python.ipynb`](./Argumentation-01b-Schemes-Walton-Python.ipynb) reste la provenance : sa table, ses 33 ensembles de mots-clés et ses textes de banc sont **cités** ici. Le module `argumentation_schemes.py` (stdlib pure) est **réimplanté en C#** dans la première cellule de code — BCL seulement, aucun NuGet, aucun LLM, aucun aléatoire. Le carnet rejoue les mêmes entrées et atteste les mêmes valeurs par des assertions exécutables, et l'échec reste **bruyant** : `null` signifie honnêtement « aucun schéma détecté », jamais une étiquette fabriquée.

**Provenance et contrat de fidélité.** La table et les forces a priori viennent verbatim du livrable étudiant `1_2_7_argumentation_dialogique` ; les questions critiques sont l'ajout canonique du tronc EPITA (le moteur étudiant n'en portait aucune). Le port documente les **divergences mesurées sur le source** au lieu de les corriger en silence : la docstring du source annonce un vocabulaire « FR + EN » alors que les 33 ensembles mesurés sont exclusivement français accentués (§4) ; la règle de la paire complète connait un singleton assumé (§6) ; et « donc » arme `cause_effect` avant `modus_ponens` (§5) — trois bornes du moteur d'origine, portées telles quelles et rendues visibles. La règle du dépôt s'applique : on restaure, on ne fabrique pas.

**Série Argumentation (C#)** | [← 01-Toulmin-Model-CSharp](./Argumentation-01-Toulmin-Model-CSharp.ipynb) | Jumeau Python : [01b-Schemes-Walton-Python](./Argumentation-01b-Schemes-Walton-Python.ipynb) | Suivant en C# : [02b-Argumentum-Cards-CSharp](./Argumentation-02b-Argumentum-Cards-CSharp.ipynb) | [↑ Série Argumentation](README.md)

## §1 — Le modèle : dix schémas, une force a priori, des questions critiques

Chaque schéma est un **gabarit** : des prémisses types qui mènent à une conclusion type. La propriété `Strength` est la **force a priori** du moteur étudiant d'origine — un a priori sur la fiabilité du schéma, *pas* un score mesuré sur le texte. Les `CriticalQuestions` sont les questions canoniques de Walton : ce qu'un challenger demande pour éprouver le schéma.

In [1]:
// Le moteur du jumeau Python (`argumentation_schemes.py`, stdlib pure) est réimplanté ici
// en C# : mêmes dix schémas, mêmes forces a priori, mêmes 33 ensembles de mots-clés
// français accentués, même ordre canonique de départage. BCL seulement, aucun NuGet.
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;
using System.Text;

/// <summary>Un schéma d'argumentation de type Walton.</summary>
public sealed class ArgumentationScheme
{
    public string Key { get; }
    public string Label { get; }
    public IReadOnlyList<string> PremisesPattern { get; }
    public string ConclusionPattern { get; }
    public double Strength { get; }
    public IReadOnlyList<string> CriticalQuestions { get; }

    public ArgumentationScheme(
        string key,
        string label,
        string[] premisesPattern,
        string conclusionPattern,
        double strength,
        params string[] criticalQuestions)
    {
        Key = key;
        Label = label;
        PremisesPattern = premisesPattern;
        ConclusionPattern = conclusionPattern;
        Strength = strength;
        CriticalQuestions = criticalQuestions;
    }
}

/// <summary>Table des dix schémas, classifieur lexical déterministe, rapport d'introspection.</summary>
public static class Schemes
{
    // Ordre d'insertion du source Python (dict de _load_argumentation_schemes) : il fixe
    // l'ordre d'affichage de la table et celui des dix lignes du rendu textuel (§8).
    public static readonly IReadOnlyList<ArgumentationScheme> All = new List<ArgumentationScheme>
    {
        new ArgumentationScheme("modus_ponens", "Déduction (modus ponens)",
            new[] { "P", "P → Q" }, "Q", 1.0,
            "La prémisse P est-elle effectivement établie ?",
            "L'implication P → Q est-elle valide (pas un sophisme conditionnel) ?"),
        new ArgumentationScheme("expert_opinion", "Argument d'autorité (advice of an expert)",
            new[] { "Expert E says P", "E is expert in domain D", "P is in domain D" }, "P", 0.8,
            "E est-elle réellement une source experte sur ce domaine ?",
            "L'avis de E est-il cohérent avec le consensus des autres experts ?",
            "Y a-t-il une preuve directe (au-delà du seul témoignage) ?"),
        new ArgumentationScheme("analogy", "Argument par analogie",
            new[] { "Case A has property X", "Case B is similar to A", "X is relevant" },
            "Case B has property X", 0.6,
            "En quoi les cas A et B sont-ils réellement similaires sur la dimension pertinente ?",
            "Existe-t-il une différence pertinente qui brise l'analogie ?"),
        new ArgumentationScheme("cause_effect", "Argument de cause à effet",
            new[] { "A causes B", "A occurred" }, "B will occur", 0.7,
            "La relation causale A → B est-elle établie (et non une simple corrélation) ?",
            "Y a-t-il d'autres causes possibles de B ?"),
        new ArgumentationScheme("consensus", "Argument d'ad hominem consensuel (appeal to consensus)",
            new[] { "Majority of experts agree on P", "Experts are qualified", "No systematic bias" },
            "P is likely true", 0.85,
            "Le consensus est-il largement partagé (et non une minorité bruyante) ?",
            "Les experts sont-ils exempts de biais systématiques ?"),
        new ArgumentationScheme("empirical_evidence", "Argument empirique (from evidence)",
            new[] { "Data shows P", "Data is reliable", "Sample is representative" },
            "P is supported by evidence", 0.9,
            "Les données sont-elles fiables (collecte, mesure) ?",
            "L'échantillon est-il représentatif de la population visée ?"),
        new ArgumentationScheme("economic_argument", "Argument économique (cost-benefit)",
            new[] { "Action A costs X", "Action A benefits Y", "Y > X" },
            "Action A is economically justified", 0.75,
            "Le calcul coût/bénéfice inclut-il tous les coûts externes ?",
            "Les bénéfices Y sont-ils réellement supérieurs aux coûts X une fois actualisés ?"),
        new ArgumentationScheme("precautionary_principle", "Principe de précaution",
            new[] { "Risk R is possible", "R has severe consequences", "Prevention is possible" },
            "Prevention should be taken", 0.7,
            "Le risque R est-il suffisamment plausible (et non spéculatif) ?",
            "Le coût de la prévention est-il proportionné à la gravité de R ?"),
        new ArgumentationScheme("moral_argument", "Argument moral (from rights)",
            new[] { "Action A affects group G", "G has rights", "A violates rights" },
            "Action A is morally wrong", 0.8,
            "L'action A viole-t-elle réellement un droit de G ?",
            "Existe-t-il un droit concurrent qui justifierait A ?"),
        new ArgumentationScheme("historical_precedent", "Argument à partir d'un précédent historique",
            new[] { "Situation S occurred before", "S led to outcome O", "Current situation similar to S" },
            "Outcome O is likely", 0.65,
            "La situation actuelle est-elle réellement analogue à S ?",
            "Les conditions causales de O en S sont-elles réunies aujourd'hui ?"),
    };

    private static readonly Dictionary<string, ArgumentationScheme> ByKey =
        All.ToDictionary(s => s.Key);

    /// <summary>Le schéma d'une clé de la table.</summary>
    public static ArgumentationScheme Get(string key) => ByKey[key];

    // Ordre canonique de départage : les schémas spécifiques d'abord, modus_ponens en
    // dernier — « donc » est partout, ce signal ne doit tirer que quand rien de plus
    // spécifique n'a tiré.
    public static readonly IReadOnlyList<string> CanonicalOrder = new[]
    {
        "expert_opinion",
        "empirical_evidence",
        "consensus",
        "analogy",
        "cause_effect",
        "economic_argument",
        "precautionary_principle",
        "moral_argument",
        "historical_precedent",
        "modus_ponens",
    };

    // 33 ensembles de mots-clés (sous-chaînes en minuscules). Un ensemble ne tire que si
    // TOUS ses mots apparaissent (AND intra-ensemble, précision privilégiée sur le rappel).
    // Vocabulaire exclusivement français accentué : un texte désaccentué rend null (§4).
    // Deux écarts mesurés sur le source, portés tels quels : « principe de précaution » est
    // un singleton, et « donc » arme aussi cause_effect avant modus_ponens dans l'ordre.
    private static readonly Dictionary<string, string[][]> Keywords = new()
    {
        ["expert_opinion"] = new[]
        {
            new[] { "expert", "domaine" },
            new[] { "selon", "spécialiste" },
            new[] { "autorité", "compétent" },
            new[] { "source", "expert" },
        },
        ["analogy"] = new[]
        {
            new[] { "analogue", "similaire" },
            new[] { "comme", "semblable" },
            new[] { "à l'instar", "comparable" },
        },
        ["cause_effect"] = new[]
        {
            new[] { "cause", "effet" },
            new[] { "entraîne", "conséquence" },
            new[] { "provoque", "résultat" },
            new[] { "donc", "parce que" },
        },
        ["consensus"] = new[]
        {
            new[] { "consensus", "experts" },
            new[] { "majorité", "accord" },
            new[] { "unanime", "scientifiques" },
        },
        ["empirical_evidence"] = new[]
        {
            new[] { "données", "représentatif" },
            new[] { "étude", "mesure" },
            new[] { "statistique", "échantillon" },
            new[] { "résultat", "expérience" },
        },
        ["economic_argument"] = new[]
        {
            new[] { "coût", "bénéfice" },
            new[] { "économique", "rentable" },
            new[] { "investissement", "retour" },
        },
        ["precautionary_principle"] = new[]
        {
            new[] { "risque", "prévention" },
            new[] { "principe de précaution" },
            new[] { "danger", "éviter" },
        },
        ["moral_argument"] = new[]
        {
            new[] { "droit", "violation" },
            new[] { "morale", "devoir" },
            new[] { "éthique", "injuste" },
            new[] { "droits", "atteinte" },
        },
        ["historical_precedent"] = new[]
        {
            new[] { "précédent", "historique" },
            new[] { "déjà", "passé" },
            new[] { "autrefois", "abouti" },
        },
        ["modus_ponens"] = new[]
        {
            new[] { "donc", "implique" },
            new[] { "par conséquent", "si" },
        },
    };

    /// <summary>Nombre total d'ensembles de mots-clés de la table (33 au source).</summary>
    public static int KeywordSetCount => Keywords.Values.Sum(sets => sets.Length);

    private static bool Tire(string lowered, string[] kset) =>
        kset.All(kw => lowered.Contains(kw, StringComparison.Ordinal));

    /// <summary>
    /// Classe un texte vers son schéma, de façon déterministe : le premier schéma dont un
    /// ensemble de mots-clés tire, dans l'ordre canonique. Rend null quand rien ne tire —
    /// un « aucun schéma détecté » honnête, jamais une étiquette fabriquée.
    /// </summary>
    public static ArgumentationScheme Classify(string text)
    {
        if (string.IsNullOrEmpty(text)) return null;
        var lowered = text.ToLowerInvariant();
        foreach (var key in CanonicalOrder)
        {
            if (!Keywords.TryGetValue(key, out var sets)) continue;
            foreach (var kset in sets)
            {
                if (Tire(lowered, kset)) return Get(key);
            }
        }
        return null;
    }

    /// <summary>
    /// Rapport d'introspection : chaque schéma qui tire sur le texte, dans l'ordre canonique,
    /// avec l'ensemble de mots-clés qui a tiré et son rang. Le premier élément est toujours
    /// le schéma que <see cref="Classify"/> rendrait (quand la liste est non vide).
    /// </summary>
    public static List<(string Key, string[] MatchedKeywords, int Rank)> MatchReport(string text)
    {
        var report = new List<(string Key, string[] MatchedKeywords, int Rank)>();
        if (string.IsNullOrEmpty(text)) return report;
        var lowered = text.ToLowerInvariant();
        for (var i = 0; i < CanonicalOrder.Count; i++)
        {
            var key = CanonicalOrder[i];
            if (!Keywords.TryGetValue(key, out var sets)) continue;
            foreach (var kset in sets)
            {
                if (Tire(lowered, kset))
                {
                    report.Add((key, kset, i + 1));
                    break;
                }
            }
        }
        return report;
    }

    /// <summary>
    /// Rend la table comme bloc de connaissance textuel, borné à <paramref name="limit"/>
    /// schémas ; chaque ligne porte deux questions critiques (troncature du source, §8).
    /// </summary>
    public static string AsPromptContext(int limit = 10)
    {
        var lines = new List<string>();
        var i = 0;
        foreach (var s in All.Take(limit))
        {
            i++;
            var qs = string.Join(" ; ", s.CriticalQuestions.Take(2));
            lines.Add($"  {i}. « {s.Label} » (force a priori "
                + s.Strength.ToString("0.00", CultureInfo.InvariantCulture)
                + $" — questions critiques de test : {qs}");
        }
        return string.Join("\n", lines);
    }
}

/// <summary>Assertions exécutables du carnet : une valeur qui diffère lève.</summary>
public static class Verif
{
    public static void Exiger(bool condition, string message)
    {
        if (!condition) throw new InvalidOperationException(message);
    }
}

var culture = CultureInfo.InvariantCulture;
Console.WriteLine($"Moteur C# prêt : {Schemes.All.Count} schémas, "
    + $"{Schemes.KeywordSetCount} ensembles de mots-clés.");

Moteur C# prêt : 10 schémas, 33 ensembles de mots-clés.


In [2]:
// La table : clé, force a priori, libellé, nombre de questions critiques.
Console.WriteLine($"{Schemes.All.Count} schémas chargés.\n");
Console.WriteLine($"{"clé",-24}{"force",6}   {"label",-46}{"questions",10}");
Console.WriteLine(new string('-', 90));
foreach (var s in Schemes.All)
{
    var force = s.Strength.ToString("0.00", CultureInfo.InvariantCulture);
    Console.WriteLine($"{s.Key,-24}{force,6}   {s.Label,-46}{s.CriticalQuestions.Count,10}");
}

10 schémas chargés.



clé                      force   label                                          questions


------------------------------------------------------------------------------------------


modus_ponens              1.00   Déduction (modus ponens)                               2


expert_opinion            0.80   Argument d'autorité (advice of an expert)              3


analogy                   0.60   Argument par analogie                                  2


... (3 outputs left out)

economic_argument         0.75   Argument économique (cost-benefit)                     2


precautionary_principle   0.70   Principe de précaution                                 2


moral_argument            0.80   Argument moral (from rights)                           2


historical_precedent      0.65   Argument à partir d'un précédent historique            2


**Lecture du résultat.** La table va de la déduction (`modus_ponens`, force `1.00` — le seul schéma conclusif : si les prémisses tiennent, la conclusion tient) à l'analogie (`analogy`, `0.60` — la forme la plus fragile du banc : la similitude n'entraine jamais la propriété par nécessité) : l'ordre des forces a priori épouse l'intuition argumentative classique, l'argument empirique (`0.90`) et le consensus d'experts (`0.85`) tenant le haut du pavé non déductif. Deux lectures d'attention : d'abord, la `Strength` est un **a priori déclaré par le moteur étudiant**, pas une statistique mesurée sur un corpus — elle fixe l'ordre de confiance initial, pas un verdict ; ensuite, la colonne `questions` compte 2 questions critiques partout sauf l'autorité (`expert_opinion`, 3) : l'argument d'autorité est le seul du banc à exiger trois tests — expertise réelle, cohérence avec les pairs, preuve directe au-delà du témoignage. Le nombre de questions est lui-même une théorie de la fragilité : plus un schéma est testable, plus sa force dépend des réponses.

In [3]:
// Un schéma complet : prémisses types, conclusion type, questions critiques.
var s = Schemes.Get("expert_opinion");
Console.WriteLine($"Label : {s.Label}");
Console.WriteLine($"Force a priori : {s.Strength.ToString(CultureInfo.InvariantCulture)}");
Console.WriteLine("\nPrémisses types :");
for (var i = 0; i < s.PremisesPattern.Count; i++)
    Console.WriteLine($"  P{i + 1}. {s.PremisesPattern[i]}");
Console.WriteLine($"\nConclusion type : {s.ConclusionPattern}");
Console.WriteLine("\nQuestions critiques (le test du challenger) :");
for (var i = 0; i < s.CriticalQuestions.Count; i++)
    Console.WriteLine($"  CQ{i + 1}. {s.CriticalQuestions[i]}");

Label : Argument d'autorité (advice of an expert)


Force a priori : 0.8



Prémisses types :


  P1. Expert E says P


  P2. E is expert in domain D


  P3. P is in domain D



Conclusion type : P



Questions critiques (le test du challenger) :


  CQ1. E est-elle réellement une source experte sur ce domaine ?


  CQ2. L'avis de E est-il cohérent avec le consensus des autres experts ?


  CQ3. Y a-t-il une preuve directe (au-delà du seul témoignage) ?


**Lecture du résultat.** Le gabarit de l'autorité se lit comme une checklist : trois prémisses (`E affirme P`, `E est experte du domaine D`, `P appartient à D`) dont la conjonction autorise la conclusion `P`. Classer un texte `expert_opinion` ne dit **pas** que l'argument est bon — cela dit de *quelle forme* il est, donc de quel test il doit passer : la question CQ3 (« y a-t-il une preuve directe au-delà du seul témoignage ? ») est exactement ce qui distingue un appel d'autorité légitime d'un sophisme d'autorité.

## §2 — Classifications réelles : le matcher départage dix textes

Le banc ci-dessous (verbatim du source EPITA, cité du jumeau Python) porte **un exemple par schéma**. Chaque texte est soumis au classifieur réel `Schemes.Classify` ; l'assertion de round-trip vérifie que le schéma attendu est bien celui qui est rendu — le carnet ne peut pas dériver silencieusement.

In [4]:
// Banc de dix textes, un par schéma de la table : l'attendu est la clé que
// Schemes.Classify doit rendre pour ce texte, contrôlée en aller-retour.
var banc = new (string Texte, string Attendu)[]
{
    ("Selon un spécialiste reconnu, la molécule est sans danger pour l'usage courant.", "expert_opinion"),
    ("L'étude repose sur une mesure directe et un échantillon représentatif du public.", "empirical_evidence"),
    ("La majorité des chercheurs parviennent à un accord sur ce point.", "consensus"),
    ("À l'instar du pilote maritime, comparable à un navigateur solitaire, le capitaine doit tracer sa route.", "analogy"),
    ("Cette cause profonde produit un effet mesurable sur le rendement.", "cause_effect"),
    ("Le coût total reste faible au regard du bénéfice annuel obtenu.", "economic_argument"),
    ("Compte tenu du risque identifié, la prévention s'impose dès maintenant.", "precautionary_principle"),
    ("Cette atteinte aux droits fondamentaux constitue une violation caractérisée.", "moral_argument"),
    ("Il existe un précédent historique : la même configuration a déjà été observée.", "historical_precedent"),
    ("La règle l'implique ; par conséquent, si la prémisse est établie, la conclusion suit.", "modus_ponens"),
};

var ok = 0;
foreach (var (texte, attendu) in banc)
{
    var scheme = Schemes.Classify(texte);
    Verif.Exiger(scheme != null && scheme.Key == attendu,
        $"attendu {attendu}, obtenu {(scheme == null ? "null" : scheme.Key)} sur « {texte} »");
    ok++;
    // Troncature à 62 caractères : le banc tient en dix lignes de console lisibles.
    var extrait = texte.Length <= 62 ? texte : texte.Substring(0, 62);
    Console.WriteLine($"{attendu,-24} <- {extrait}");
}
Console.WriteLine($"\n{ok}/10 exemples classés selon l'attendu (round-trip vérifié).");

expert_opinion           <- Selon un spécialiste reconnu, la molécule est sans danger pour


empirical_evidence       <- L'étude repose sur une mesure directe et un échantillon représ


consensus                <- La majorité des chercheurs parviennent à un accord sur ce poin


analogy                  <- À l'instar du pilote maritime, comparable à un navigateur soli


cause_effect             <- Cette cause profonde produit un effet mesurable sur le rendeme


economic_argument        <- Le coût total reste faible au regard du bénéfice annuel obtenu


precautionary_principle  <- Compte tenu du risque identifié, la prévention s'impose dès ma


moral_argument           <- Cette atteinte aux droits fondamentaux constitue une violation


historical_precedent     <- Il existe un précédent historique : la même configuration a dé


modus_ponens             <- La règle l'implique ; par conséquent, si la prémisse est établ



10/10 exemples classés selon l'attendu (round-trip vérifié).


**Lecture du résultat.** Dix textes, dix schémas, aucune erreur : chaque texte du banc porte la **paire complète** de mots-clés qui signe son schéma — « selon + spécialiste » pour l'autorité, « étude + mesure » pour l'empirique, « à l'instar + comparable » pour l'analogie. La paire joue le rôle d'une signature : chacun des mots seul est banal (« mesure » apparaît dans une recette de cuisine, « spécialiste » dans une annonce de conférence), leur co-occurrence dans un même texte est le signal argumentatif. Remarquez la prudence du banc pour `modus_ponens` : son exemple utilise « par conséquent + si » et évite soigneusement « donc », dont on verra au §5 qu'il arme aussi `cause_effect` — le banc contourne l'écueil au lieu de le cacher, et l'exercice 3 vous demande de le ré-armer vous-même. Dernier point de mesure : ce banc n'exerce qu'**un ensemble de mots-clés par schéma** (10 des 33 du moteur, ~30 %) — toujours le même profil de paire canonique ; les paires alternatives sont laissées à l'exercice 1, et les tests automatisés de l'organe en couvrent d'autres.

## §3 — L'honnêteté du `null` : quatre textes sans schéma

Le matcher exige la **paire complète** : un mot-clé isolé ne tire jamais. Quand rien ne tire, le classifieur rend `null` — un « aucun schéma détecté » **honnête**, jamais une étiquette fabriquée. C'est le contrat d'échec bruyant de l'organe.

In [5]:
// Contrôle négatif : quatre textes sans argument (salutation, inventaire,
// annonce, chaîne vide) doivent rendre null. Sans ce contrôle, un classifieur
// qui tire toujours passerait le banc positif sans être vu.
var negatifs = new[]
{
    "Bonjour, il fait beau aujourd'hui.",
    "Le rapport compte douze pages et trois annexes.",
    "L'expert est attendu à la conférence.",
    "",
};

foreach (var texte in negatifs)
{
    var scheme = Schemes.Classify(texte);
    Verif.Exiger(scheme == null, $"attendu null, obtenu {scheme?.Key} sur « {texte} »");
    Console.WriteLine($"null <- \"{texte}\"");
}

Console.WriteLine("\n4/4 négatifs rendent null (honnêteté du classifieur).");

null <- "Bonjour, il fait beau aujourd'hui."


null <- "Le rapport compte douze pages et trois annexes."


null <- "L'expert est attendu à la conférence."


null <- ""



4/4 négatifs rendent null (honnêteté du classifieur).


**Lecture du résultat.** Le troisième négatif est le plus instructif : « L'**expert** est attendu à la conférence » contient bien un mot-clé du schéma d'autorité — mais isolé, sans son binôme (« domaine », « spécialiste », « source »...). Le classifieur préfère ne rien dire plutôt que de mal étiqueter : c'est le choix **précision d'abord**, assumé dans la conception. La conséquence opérationnelle est à écrire noir sur blanc : sur un corpus de textes neutres (météo, comptes rendus, petites annonces), ce classifieur rendra majoritairement `null` — et c'est le comportement **voulu**. Un outil de classification d'argument qui étiquetterait « il fait beau » comme analogie serait inutilisable ; l'honnêteté du `null` est ce qui rend les étiquettes positives crédibles quand elles arrivent. Le texte vide, lui, rend `null` immédiatement, sans même parcourir la table — la garde minimale avant tout parcours.

## §4 — La limite assumée : les mots-clés sont accentués

Les 33 ensembles de mots-clés de la table sont exclusivement **français accentués**. La docstring du source original annonce un vocabulaire « FR + EN » — mesure faite, aucun mot anglais ne figure dans la table, et un texte désaccentué casse la paire. Le port documente la limite au lieu de la corriger : c'est une borne du moteur d'origine, pas un défaut du port.

In [6]:
// Texte désaccentué : la paire « selon + spécialiste » ne se reforme pas.
var texteDesaccentue = "Selon ce specialiste reconnu, la molecule est sans danger.";
var classe = Schemes.Classify(texteDesaccentue);
Console.WriteLine($"Schemes.Classify(texteDesaccentue) = {(classe == null ? "null" : classe.Key)}");
Verif.Exiger(classe == null, "un texte désaccentué ne doit pas reformer la paire");

// Mot isolé : jamais de tir, quelle que soit la paire dont il est extrait.
foreach (var mot in new[] { "donc", "si", "comme", "expert", "droit", "mesure" })
{
    Verif.Exiger(Schemes.Classify(mot) == null, $"le mot isolé « {mot} » ne doit pas tirer seul");
}
Console.WriteLine("6 mots isolés testés : aucun ne tire seul.");

Schemes.Classify(texteDesaccentue) = null


6 mots isolés testés : aucun ne tire seul.


**Lecture du résultat.** « specialiste » sans accent ne contient pas la sous-chaîne « spécialiste » : la paire ne se reforme pas et le texte rend `null`. C'est la contrepartie du choix de sous-chaînes accentuées — le même mécanisme qui protège la précision (le §3) rend le classifieur sensible à la graphie exacte. Un pipeline amont qui normalise les accents doit en être conscient : il perd alors tout le banc.

## §5 — Le départage : l'ordre canonique et le piège de « donc »

Un texte peut armer **plusieurs** schémas à la fois. Le classifieur les départage par un **ordre canonique** : les schémas spécifiques d'abord, `modus_ponens` en dernier — parce que « donc » est partout, ce signal ne doit tirer que quand rien de plus spécifique n'a tiré. La méthode `Schemes.MatchReport` expose ce mécanisme : elle rend *tous* les schémas qui tirent, dans l'ordre du départage.

In [7]:
// Texte ambigu : arme expert_opinion ET modus_ponens.
var ambigu = "Selon ce spécialiste, la règle implique donc la conclusion.";
var report = Schemes.MatchReport(ambigu);
foreach (var (key, matched, rank) in report)
{
    var liste = "[" + string.Join(", ", matched.Select(m => "'" + m + "'")) + "]";
    Console.WriteLine($"rang {rank,2} : {key,-18} armé par {liste}");
}
var retenu = Schemes.Classify(ambigu);
Console.WriteLine($"\nSchemes.Classify retient : {(retenu == null ? "null" : retenu.Key)}");
Verif.Exiger(retenu != null && retenu.Key == "expert_opinion", "l'autorité doit l'emporter au rang 1");
Verif.Exiger(report.Select(r => r.Key).SequenceEqual(new[] { "expert_opinion", "modus_ponens" }),
    "le rapport doit porter exactement les deux candidats, dans cet ordre");

// « donc » arme AUSSI cause_effect (paire « donc + parce que »), placé AVANT modus_ponens.
var piege = "Il pleut parce que le nuage est là, donc le sol est humide.";
var classePiege = Schemes.Classify(piege);
Console.WriteLine($"\nPiège cause/modus : Schemes.Classify retient {(classePiege == null ? "null" : classePiege.Key)}");
Verif.Exiger(classePiege != null && classePiege.Key == "cause_effect",
    "la paire « donc + parce que » arme cause_effect avant modus_ponens");

rang  1 : expert_opinion     armé par ['selon', 'spécialiste']


rang 10 : modus_ponens       armé par ['donc', 'implique']



Schemes.Classify retient : expert_opinion



Piège cause/modus : Schemes.Classify retient cause_effect


**Lecture du résultat.** Le rapport montre deux candidats sur le texte ambigu : l'autorité (rang 1) et le modus ponens (rang 10) — et le classifieur retient l'autorité, la plus spécifique. `Schemes.MatchReport` n'est pas un second classifieur : c'est la **même mécanique exposée** (le premier élément du rapport est toujours ce que `Schemes.Classify` rend), un instrument d'inspection qui rend visible un choix sinon invisible. Le second cas révèle la mesure fine de l'ordre : la paire « donc + parce que » arme `cause_effect`, placé *avant* `modus_ponens` — un texte causal-déductif est donc classé causal, et le modus ponens lexical ne tire que sur ses propres paires (« donc + implique », « par conséquent + si »). L'ordre canonique n'est pas une préférence esthétique : c'est lui qui décide de l'étiquette finale dès qu'un texte arme plusieurs schémas — et une inversion de cet ordre changerait des étiquettes sans toucher un seul mot-clé.

## §6 — L'unique singleton : « principe de précaution »

La règle du §3 (paire complète) connait **un seul écart**, mesuré sur le source : l'ensemble `["principe de précaution"]` — la locution entière suffit à armer le schéma, sans binôme. Écart porté tel quel : il est volontaire (une locution figée est aussi discriminante qu'une paire) et documenté ici au lieu d'être corrigé.

In [8]:
var texte = "Nous invoquons le principe de précaution.";
var scheme = Schemes.Classify(texte);
Console.WriteLine($"Schemes.Classify retient : {(scheme == null ? "null" : scheme.Key)}");
Verif.Exiger(scheme != null && scheme.Key == "precautionary_principle", "le singleton doit tirer");
Console.WriteLine("La locution entière tire seule : l'unique singleton de la table (33 ensembles, 32 paires, 1 singleton).");

Schemes.Classify retient : precautionary_principle


La locution entière tire seule : l'unique singleton de la table (33 ensembles, 32 paires, 1 singleton).


**Lecture du résultat.** Le singleton passe parce que la locution « principe de précaution » est **figée** : sa présence dans un texte est déjà un signal argumentatif complet, là où « expert » seul ne dit rien (le mot apparaît dans une annonce de conférence, cf. §3). La table encode ainsi une gradation implicite : plus le mot-clé est banal, plus il exige de compagnie.

## §7 — Les questions critiques : de la classification au test

Classer **nomme** le schéma ; les questions critiques le **testent**. C'est la brique qui manquait au moteur étudiant d'origine — la table portait les forces mais aucune question : le tronc EPITA y a ajouté les questions canoniques de Walton. L'audit ci-dessous montre l'enchaînement complet sur un texte réel : forme reconnue, tests applicables, force a priori.

In [9]:
// Audit d'un argument retenu : les questions critiques (CQ) que le schéma
// impose de vérifier avant d'accepter la conclusion, affichées une par une.
var texteAudit = "L'étude repose sur une mesure directe et un échantillon représentatif du public.";
var schemeAudit = Schemes.Classify(texteAudit);
Verif.Exiger(schemeAudit != null && schemeAudit.Key == "empirical_evidence", "l'argument empirique doit être reconnu");
Console.WriteLine($"Texte classé : {schemeAudit.Label} "
    + $"(force a priori {schemeAudit.Strength.ToString("0.00", CultureInfo.InvariantCulture)})");
Console.WriteLine("\nAudit — les tests que ce texte doit passer :");
for (var i = 0; i < schemeAudit.CriticalQuestions.Count; i++)
    Console.WriteLine($"  CQ{i + 1}. {schemeAudit.CriticalQuestions[i]}");

Texte classé : Argument empirique (from evidence) (force a priori 0.90)



Audit — les tests que ce texte doit passer :


  CQ1. Les données sont-elles fiables (collecte, mesure) ?


  CQ2. L'échantillon est-il représentatif de la population visée ?


**Lecture du résultat.** L'argument empirique est reconnu à sa paire « étude + mesure », et l'audit liste ses deux questions canoniques : fiabilité des données, représentativité de l'échantillon. Un texte peut porter la *forme* parfaite d'un argument empirique et échouer à ses deux tests (données falsifiées, échantillon biaisé) — la classification ouvre l'évaluation, elle ne la remplace pas. C'est la même discipline que la matrice de richesse formelle (`Formal_Richness_Matrix`) appliquée cette fois à l'argumentation informelle : nommer la forme, puis demander compte. Et le contraste avec le détecteur de sophismes de la série prend ici son sens : détecter un sophisme dit ce qui *cloche* ; classifier un schéma dit quelle *force* l'argument prétend avoir — les deux questions ne se confondent pas, un argument d'autorité parfaitement formel reste un argument d'autorité, à tester comme tel. Le lien avec les dialogues protocolisés est le même : les protocoles d'inquiry/persuasion organisent *qui parle quand*, les schémas organisent *ce que vaut chaque coup*.

## §8 — La table comme contexte : le rendu textuel

Côté EPITA, le débat multi-agents consomme la table via `Schemes.AsPromptContext` : un rendu textuel borné que le prompt LLM reçoit pour ancrer ses échanges sur des schémas réels. Côté CoursIA, ce rendu sert de lecture condensée de la table — et en révèle une mesure fine : la troncature à deux questions par schéma.

In [10]:
var ctx = Schemes.AsPromptContext();
Console.WriteLine($"{ctx.Split('\n').Length} lignes rendues :");
Console.WriteLine(ctx);
Verif.Exiger(ctx.Split('\n').Length == 10, "le rendu doit porter dix lignes, une par schéma");

10 lignes rendues :


  1. « Déduction (modus ponens) » (force a priori 1.00 — questions critiques de test : La prémisse P est-elle effectivement établie ? ; L'implication P → Q est-elle valide (pas un sophisme conditionnel) ?
  2. « Argument d'autorité (advice of an expert) » (force a priori 0.80 — questions critiques de test : E est-elle réellement une source experte sur ce domaine ? ; L'avis de E est-il cohérent avec le consensus des autres experts ?
  3. « Argument par analogie » (force a priori 0.60 — questions critiques de test : En quoi les cas A et B sont-ils réellement similaires sur la dimension pertinente ? ; Existe-t-il une différence pertinente qui brise l'analogie ?
  4. « Argument de cause à effet » (force a priori 0.70 — questions critiques de test : La relation causale A → B est-elle établie (et non une simple corrélation) ? ; Y a-t-il d'autres causes possibles de B ?
  5. « Argument d'ad hominem consensuel (appeal to consensus) » (force a priori 0.85 — questions critiques de test : Le cons

**Lecture du résultat.** Le rendu porte les dix schémas avec force a priori et deux questions chacun — mais la **troisième** question critique de l'autorité (« y a-t-il une preuve directe au-delà du seul témoignage ? ») n'apparaît pas : le rendu du source tronque aux deux premières, porté tel quel. Mesure visible dans le carnet lui-même : la table du §1 compte `questions: 3` pour l'autorité, ce rendu n'en montre que deux. Dans le pipeline LLM d'origine, cette économie de budget de prompt se paye d'un test perdu — leçons des deux côtés du pont.

## Exercice 1 — Armer une paire alternative

La table d'autorité compte **quatre** ensembles de mots-clés ; le banc du §2 n'en exerce qu'un (« selon + spécialiste »). Écrivez dans `monTexte` un texte court qui arme la paire `["source", "expert"]` — le classifieur doit rendre `expert_opinion`. *Indice : les deux mots doivent apparaître, n'importe où dans le texte.*

In [11]:
// Exercice à compléter : la paire « source + expert » doit classer le texte en autorité.
var monTexte = "";  // TODO étudiant : votre texte ici.

var resultat = Schemes.Classify(monTexte);
Console.WriteLine($"Schemes.Classify(monTexte) = {(resultat == null ? "null" : resultat.Key)}");

Schemes.Classify(monTexte) = null


## Exercice 2 — Un négatif résistant

Le §3 a montré qu'un mot isolé ne tire jamais. Compliquons : écrivez dans `monDefi` un texte qui contient **un mot-clé de deux schémas différents** (par exemple « expert » et « mesure ») et qui pourtant rend `null`. *Indice : chaque paire exige ses DEUX mots — un mot de chaque paire ne suffit pas.*

In [12]:
// Exercice à compléter : un mot de deux schémas différents, et pourtant null.
var monDefi = "";  // TODO étudiant : votre texte ici.

var resultatDefi = Schemes.Classify(monDefi);
Console.WriteLine($"Schemes.Classify(monDefi) = {(resultatDefi == null ? "null" : resultatDefi.Key)} (attendu : null)");

Schemes.Classify(monDefi) = null (attendu : null)


## Exercice 3 — Prédire le rapport avant de le lancer

Avant d'exécuter la cellule, écrivez votre prédiction : quels schémas `Schemes.MatchReport` va-t-il lister sur le texte `monAmbigu` ci-dessous, et dans quel ordre ? Complétez `prediction` (une liste de clés), puis exécutez : la cellule affiche votre prédiction à côté du rapport réel. *Indice : repérez toutes les paires complètes présentes dans le texte, puis appliquez l'ordre canonique du §5.*

In [13]:
// Exercice à compléter : prédire les candidats AVANT l'exécution.
var monAmbigu = "Selon la source experte, le risque justifie la prévention, donc l'étude impose la mesure.";
var prediction = new List<string>();  // TODO étudiant : clés attendues, dans l'ordre du rapport.

var rapport = Schemes.MatchReport(monAmbigu);
Console.WriteLine($"Rapport réel     : [{string.Join(", ", rapport.Select(r => r.Key))}]");
Console.WriteLine($"Votre prédiction : [{string.Join(", ", prediction)}]");

Rapport réel     : [expert_opinion, empirical_evidence, precautionary_principle]


Votre prédiction : []


## À retenir

- **Dix schémas de Walton** : prémisses types, conclusion type, force a priori (1.00 pour la déduction, 0.60 pour l'analogie) et questions critiques canoniques — la table est celle du moteur étudiant EPITA, les questions sont l'ajout du tronc.
- **Un classifieur lexical déterministe** : 33 ensembles de mots-clés français accentués, paire complète exigée (un singleton : « principe de précaution »), ordre canonique qui départage — `modus_ponens` en dernier car « donc » arme aussi `cause_effect`.
- **L'échec est bruyant** : `null` est un résultat honnête, jamais une étiquette fabriquée. Un mot isolé ne tire pas ; un texte désaccentué ne tire pas.
- **Classer nomme, les questions testent** : l'audit d'un texte classé liste les tests canoniques qu'il doit passer — la classification ouvre l'évaluation, elle ne la remplace pas.
- **Position dans la série** : `Toulmin_Model` déplie la *structure* d'un argument, `Dung_AF_Semantics` calcule *qui survit* aux attaques ; ce carnet identifie *la forme stéréotypée* d'un texte et fournit son test — les trois niveaux de l'argumentation informelle à la computationnelle.
- **Les limites mesurées, portées au grand jour** : vocabulaire exclusivement français accentué (un texte désaccentué rend `null`, §4), un singleton assumé (§6), « donc » partagé entre deux schémas (§5), rendu tronqué à deux questions (§8). Aucune de ces bornes n'est un défaut du port : chacune est une mesure du moteur d'origine, documentée pour que l'utilisateur du classifieur sache exactement ce qu'il tient — et ce qu'il ne tient pas.
- **Le jumeau C#** : le moteur est BCL pure (aucun NuGet, aucun LLM), les mêmes entrées rendent les mêmes valeurs, et chaque valeur est attestée par une assertion exécutable (`Verif.Exiger`) — la parité ne repose pas sur une relecture, mais sur une exécution.

## Vérifiez votre compréhension

**Question 1 — Pourquoi « L'expert est attendu à la conférence » rend-il `null`, alors que « Selon un spécialiste reconnu, la molécule est sans danger… » classe `expert_opinion` ?**

Le §3 imprime `null <- "L'expert est attendu à la conférence."` parmi ses quatre négatifs.

<details><summary>Réponse</summary>

Parce que le classifieur exige la **paire complète** de mots-clés. `expert_opinion` est armé par `['selon', 'spécialiste']` (rang 1) : le texte négatif contient « expert » seul, un mot isolé ne tire jamais — le §4 le mesure explicitement (`6 mots isolés testés : aucun ne tire seul`), et le §2 montre le contraste : chaque texte classé porte bien les deux mots de sa paire. `null` est donc le verdict honnête d'un moteur lexical à paire exigée, pas un raté.

</details>

**Question 2 — Deux textes contiennent « donc » : l'un classe `expert_opinion`, l'autre `cause_effect`. Qu'est-ce qui départage ?**

Le §5 imprime `rang 1 : expert_opinion armé par ['selon', 'spécialiste']`, `rang 10 : modus_ponens armé par ['donc', 'implique']`, puis `Schemes.Classify retient : expert_opinion` — et sur l'autre texte, `Piège cause/modus : Schemes.Classify retient cause_effect`.

<details><summary>Réponse</summary>

L'**ordre canonique de la table**. « donc » arme à la fois `modus_ponens` et `cause_effect` : le classement parcourt les schémas par rang, et le premier schéma **entièrement armé** gagne. Le texte qui porte `['selon', 'spécialiste']` satisfait `expert_opinion` au rang 1 avant que quiconque n'inspecte « donc » ; le texte purement causal voit `cause_effect` (rang 5) l'emporter sur `modus_ponens` (rang 10). C'est un départage déterministe et reproductible — pas une préférence cachée du modèle.

</details>

**Question 3 — Pourquoi « principe de précaution » est-il qualifié d'unique singleton de la table ?**

Le §6 imprime `Schemes.Classify retient : precautionary_principle` puis `La locution entière tire seule : l'unique singleton de la table (33 ensembles, 32 paires, 1 singleton)`.

<details><summary>Réponse</summary>

La table compte 33 ensembles de mots-clés : 32 exigent **deux** mots, un seul — « principe de précaution » — tire sur sa locution entière. La locution est assez distinctive pour qu'un seul token lexical suffise à l'identifier sans faux positifs, ce qui n'est pas le cas d'un mot comme « expert » ou « donc ». C'est une asymétrie mesurée du moteur, assumée comme telle.

</details>

## Pour aller plus loin

- [Index de la sous-série ArgumentAnalysis](README.md) — la place des schémas de Walton dans le parcours.
- [Index de la série SymbolicAI](../README.md) — le fil complet, du raisonnement formel aux systèmes multi-agents.
- [Jumeau Python de ce carnet](./Argumentation-01b-Schemes-Walton-Python.ipynb) — la provenance de la table, des 33 ensembles et du banc.

La suite naturelle en C# est [Argumentation-02b-Argumentum-Cards-CSharp.ipynb](./Argumentation-02b-Argumentum-Cards-CSharp.ipynb) : les schémas classent ce qui *devrait* convaincre, les cartes d'argument nomment les pièces du dossier — les deux lectures se complètent. Pour la structure d'un argument unique, [Argumentation-01-Toulmin-Model-CSharp.ipynb](./Argumentation-01-Toulmin-Model-CSharp.ipynb) déplie les six pièces ; pour le calcul de survie sous attaques, [Argumentation-03-Dung-AF-Semantics-CSharp.ipynb](./Argumentation-03-Dung-AF-Semantics-CSharp.ipynb). Les trois exercices ci-dessus prolongent directement le travail : armer une paire alternative, fabriquer un négatif résistant, prédire un rapport avant exécution.